### Endocarditis: Training data generation

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os
from pathlib import Path
from uuid import UUID
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import MESARuntimeBatchGenerator, TrainingDataUploader
from conditionschema.prompt_builder import PromptBuilder
from conditionschema import Schema
from mesa_runtime.admin import MesaRuntimeAdmin

In [ ]:
pb = PromptBuilder()
prompt_parameters: dict[str, str] = {"condition": "infective endocarditis", "cui": "56819008"}

#### Download documents from batch

In [ ]:
gen = MESARuntimeBatchGenerator(
    system_prompt=pb.build_datagen_prompt(prompt_parameters),
    schema=Schema,
    schema_name="conditionschema",
    document_batches=["<add document batch>"],
    model_name="qwen3.5",
    runtime=MesaRuntimeAdmin(),
)

In [ ]:
gen.get_document_files_count()

#### Sanity check

##### Start batch generation

In [ ]:
run_id: UUID = gen.generate_via_batch(100, os.environ["BUCKET"])

In [ ]:
gen.get_batch_run(run_id)

##### Download and parse batch outputs

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Start datagen

In [ ]:
run_id: UUID = gen.generate_via_batch(3000, os.environ["BUCKET"])

In [ ]:
gen.get_batch_run(run_id)

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="conditionschema",
    system_prompt=pb.build_main_prompt(prompt_parameters),
    short_description="endocarditis-batch",
    long_description="Training data for Endocarditis",
    input_folder=Path("./data/datagen/samples"),
    prompt_parameters=prompt_parameters,
)